In [1]:
from pathlib import Path
import numpy as np
import xarray as xr

In [2]:
def write_ww3_boundary_spectrum(file_path, lat, lon, freq, direction, efth, time=None):
    """
    Write a single-point WW3 boundary spectrum file, in the minimal shape
    ww3_bounc.F90 actually reads.

    This is deliberately narrower than a real ww3_ounp point-output file
    (the kind NOAA's own hindcast archive ships): things like station_name,
    frequency1/frequency2 (band edges), depth, u10m/udir, curr/currdir exist
    in that format for CF/GlobWave metadata compliance, but grepping
    ww3_bounc.F90 shows none of them are ever read (no NF90_INQ_VARID call
    for any of them). Only these are:
      - dims/vars "time", "frequency", "direction" (names must match exactly,
        ww3_bounc looks them up by name via NF90_INQ_DIMID/NF90_INQ_VARID)
      - "latitude"/"longitude" (only the first value is ever read into a
        scalar, even if this were made time-varying -- so it's fine as a
        length-1 dim)
      - "efth" (or "Efth" -- ww3_bounc falls back to that capitalization if
        "efth" isn't found)
      - no "station_name" var: if present, ww3_bounc instead reads efth as
        (time, station, frequency, direction) -- but that's not a "multi
        station" capability, it's just a different dimension order for the
        same one-point behavior: that branch's NF90_GET_VAR call hardcodes
        start/count to always read station index 1 only, no matter how many
        stations the file actually declares. Omitting station_name simply
        keeps us on the branch whose dimension order matches what we write
        here, efth(time, frequency, direction, latitude, longitude).

    "_FillValue" on efth is NOT optional the way scale_factor/add_offset
    are: ww3_bounc.F90 calls CHECK_ERR() (a hard EXTCDE(59) exit) right
    after reading it, whereas a missing scale_factor/add_offset is silently
    defaulted to 1./0. (FACTOR=1., OFFSET=0. if the NF90_GET_ATT call fails).
    We write efth as plain float (not packed short), so scale_factor=1./
    add_offset=0. is the true identity transform here -- we set them
    explicitly anyway rather than lean on that default-if-missing behavior.

    Because this writes exactly one point, ww3_bounc's own fallback (see
    W3BOUNC: when only one file is listed in spec.list, NBO2==1 and its
    interpolation block is skipped entirely) applies this single spectrum
    uniformly to *every* active boundary point on the grid -- lat/lon here
    don't need to sit exactly on a boundary cell for that reason; they're
    mostly informational until you add more points.

    Time is written as plain seconds-since-epoch floats, NOT as a
    datetime64 coordinate. This was found the hard way: when time is
    datetime64, xarray's own CF encoder recomputes the "units" string at
    write time and ignores whatever we set in .encoding["units"] -- for a
    timestamp that lands exactly on midnight it collapses to a bare
    "seconds since 1990-01-01" with no "hh:mm:ss" at all. W3TIMEMD's U2D
    parses that units string by *fixed column position* (verified by
    reading the source), so a truncated string means it reads past the
    real attribute text into whatever's in the underlying buffer -- not
    reliably a crash, just undefined. Keeping time as plain floats (not
    datetime64) means xarray never touches the units/calendar attributes
    we set, so the string we write is exactly the string in the file.

    Parameters
    ----------
    file_path : str
        Output netCDF path.
    lat, lon : float
        Nominal point location, degrees.
    freq : array-like, shape (NK,)
        Frequency bins, Hz. Doesn't have to match your grid's own NK/FR1/XFR
        (WW3_interface/WW3/model/src for your grid: NK=25, FR1=0.04118 Hz,
        XFR=1.1, i.e. freq[k] = FR1 * XFR**k for k=0..24) -- ww3_bounc
        auto-remaps (the SPCONV/W3CSPC path) onto the grid's own
        discretization if these differ, so mismatching is not fatal. Match
        them only if you'd rather skip that remap step for a first test.
    direction : array-like, shape (NTH,)
        Direction bins, degrees, "coming from" convention (clockwise from
        true north) -- ww3_bounc converts this internally
        (THETA = mod(2.5*pi - deg2rad(direction), 2*pi)). Your grid uses
        NTH=24 with a 0.0 relative first-direction offset; again, only
        worth matching if you want to dodge the SPCONV remap.
    efth : array-like, shape (NT, NK, NTH)
        2D variance density spectrum per timestep, m^2 s / rad.
    time : array-like of np.datetime64, optional
        Timestamps for each efth slice. Defaults to a single arbitrary step.
        Converted to plain seconds-since-1990-01-01 floats before writing
        (see note above) -- passed in as datetime64 just for a nicer API.
    """
    freq = np.asarray(freq, dtype=np.float32)
    direction = np.asarray(direction, dtype=np.float32)
    efth = np.asarray(efth, dtype=np.float32)
    if time is None:
        time = np.array(["2020-01-01T00:00:00"], dtype="datetime64[ns]")
    else:
        time = np.asarray(time, dtype="datetime64[ns]")

    time_units = "seconds since 1990-01-01 00:00:00.0"
    epoch = np.datetime64("1990-01-01T00:00:00", "ns")
    time_seconds = (time - epoch) / np.timedelta64(1, "s")  # plain float64, not datetime64

    ds = xr.Dataset(
        data_vars={
            "efth": (
                ("time", "frequency", "direction", "latitude", "longitude"),
                efth.reshape(len(time), len(freq), len(direction), 1, 1),
                {
                    "units": "m2 s rad-1",
                    "_FillValue": np.float32(-999.9),  # required: see docstring
                    "scale_factor": np.float32(1.0),  # identity: efth is plain float
                    "add_offset": np.float32(0.0),
                },
            ),
        },
        coords={
            # units/calendar as plain attrs on a plain float array -- see
            # docstring note on why this isn't a datetime64 coordinate.
            "time": ("time", time_seconds, {"units": time_units, "calendar": "standard"}),
            "frequency": ("frequency", freq, {"units": "s-1"}),
            "direction": ("direction", direction, {"units": "degree"}),
            "latitude": ("latitude", np.array([lat], dtype=np.float32)),
            "longitude": ("longitude", np.array([lon], dtype=np.float32)),
        },
    )

    # xarray defaults to adding a _FillValue to every float variable/coord
    # unless told not to; ww3_bounc never reads _FillValue on anything but
    # efth, but there's no reason to write attributes we didn't ask for.
    no_fill = {"_FillValue": None}
    for coord in ("time", "frequency", "direction", "latitude", "longitude"):
        ds[coord].encoding.update(no_fill)

    ds.to_netcdf(file_path, mode="w", format="NETCDF4")
    return ds


def write_ww3_boundary_spectra(file_paths, lats, lons, freq, direction, efths, time=None):
    """
    Write several single-point WW3 boundary spectrum files at once.

    Thin loop over write_ww3_boundary_spectrum -- ww3_bounc has no way to
    take more than one point from a single file (see that function's
    docstring), so "multiple points" can only ever mean multiple files.
    This just saves calling it by hand for each point.

    Parameters
    ----------
    file_paths : list[str], length N
        One output netCDF path per point.
    lats, lons : array-like, length N
        Nominal location of each point, degrees.
    freq, direction : array-like
        Shared frequency/direction bins for every point -- see
        write_ww3_boundary_spectrum.
    efths : array-like, shape (N, NT, NK, NTH)
        2D spectrum per point, per timestep.
    time : array-like of np.datetime64, optional
        Shared timestamps for every point's efth slices.

    Returns
    -------
    list of the xr.Dataset written for each point, in the same order as
    file_paths.
    """
    n_points = len(file_paths)
    assert len(lats) == n_points and len(lons) == n_points and len(efths) == n_points, (
        "file_paths, lats, lons, and efths must all have the same length"
    )
    return [
        write_ww3_boundary_spectrum(fp, lat, lon, freq, direction, efth, time=time)
        for fp, lat, lon, efth in zip(file_paths, lats, lons, efths)
    ]


def write_ww3_bounc_nml(file_dir, spec_list_filename="spec.list", mode="WRITE", interp=2, verbose=1):
    """
    Write ww3_bounc.nml, the BOUND_NML namelist that drives the ww3_bounc
    preprocessor. Points BOUND%FILE at spec_list_filename (see write_spec_list).

    Parameters
    ----------
    file_dir: str
        Directory to write ww3_bounc.nml to.
    spec_list_filename: str
        Name of the spec-list file BOUND%FILE should point at.
    mode: str
        'WRITE' to build nest.ww3 from spectra files, 'READ' to diagnose an
        existing nest.ww3 instead.
    interp: int
        Interpolation method onto boundary points: 1 (nearest), 2 (linear).
    verbose: int
        Verbosity level: 0, 1, or 2.
    """
    file_dir = Path(file_dir)
    file_dir.mkdir(parents=True, exist_ok=True)

    with open(file_dir / "ww3_bounc.nml", "w") as f:
        f.write(
            "! -------------------------------------------------------------------- !\n"
            "! WAVEWATCH III - ww3_bounc.nml - Boundary input post-processing        !\n"
            "! -------------------------------------------------------------------- !\n"
            "\n"
            "! -------------------------------------------------------------------- !\n"
            "! Define the input boundaries to preprocess via BOUND_NML namelist\n"
            "! Note: When using a rotated pole WW3 grid, the input spectra are\n"
            "! always assumed to be formulated on a standard pole.\n"
            "!\n"
            "! * namelist must be terminated with /\n"
            "! * definitions & defaults:\n"
            "!     BOUND%MODE                 = 'WRITE'            ! ['WRITE'|'READ']\n"
            "!     BOUND%INTERP               = 2                  ! interpolation [1(nearest),2(linear)]\n"
            "!     BOUND%VERBOSE              = 1                  ! [0|1|2]\n"
            "!     BOUND%FILE                 = 'spec.list'        ! input _spec.nc listing file\n"
            "! -------------------------------------------------------------------- !\n"
            "&BOUND_NML\n"
            f"  BOUND%MODE                 = '{mode}'\n"
            f"  BOUND%INTERP               = {interp}\n"
            f"  BOUND%VERBOSE              = {verbose}\n"
            f"  BOUND%FILE                 = '{spec_list_filename}'\n"
            "/\n"
            "\n"
            "! -------------------------------------------------------------------- !\n"
            "! WAVEWATCH III - end of namelist                                      !\n"
            "! -------------------------------------------------------------------- !\n"
        )


def write_spec_list(file_dir, spectra_paths, spec_list_filename="spec.list"):
    """
    Write the spec.list file that ww3_bounc.nml's BOUND%FILE points at: one
    _spec.nc path per line.

    Parameters
    ----------
    file_dir: str
        Directory to write spec_list_filename to.
    spectra_paths: list[str]
        Paths to the per-boundary-point spectra netCDF files.
    spec_list_filename: str
        Name of the file to write.
    """
    file_dir = Path(file_dir)
    file_dir.mkdir(parents=True, exist_ok=True)

    with open(file_dir / spec_list_filename, "w") as f:
        for p in spectra_paths:
            f.write(f"{p}\n")

In [ ]:
# Tsunami-pulse test, take 2: multi-time-record boundary spectra.
#
# Take 1 failed invisibly because nest.ww3 had a single time record
# (2020-01-01 00:00): the model applied it for exactly one step, then hit
# EOF looking for the next record at 00:30 and set FLBPI=.FALSE. --
# permanently disabling (and zeroing) the boundary ring for the remaining
# 30 days (w3iobcmd.F90 EOF branch; the 'L' flag in wav.log's 'b' column).
#
# So the time axis here must SPAN THE RUN: WW3 interpolates linearly
# between records (spacing need not be uniform), and only shuts boundary
# forcing off when model time passes the final record. Design:
#   - hourly records for the first 12 h: pulse at the west point for 6 h,
#     then down to calm,
#   - daily calm records out past the run end (Feb 1) so FLBPI never
#     trips EOF mid-run and the boundary stays prescribed-calm instead of
#     going dead.
#
# Two real corners of this grid's southern boundary row (from
# croc_input/ww3.obc.1/ocnice/*_{x,y}.inp: lon 328.05-333.95, lat 36.05-40.95):
# west corner gets the pulse, east corner stays calm, and with two files in
# spec.list ww3_bounc's real interpolation path (NBO2>1) fills the boundary
# cells between them.

# Match the grid's own spectral discretization (ww3_grid.inp:
# XFR=1.1, FR1=0.04118, NK=25, NTH=24) so ww3_bounc doesn't need to SPCONV-remap.
NK, NTH = 25, 24
freq = 0.04118 * 1.1 ** np.arange(NK)
direction = np.linspace(0, 360, NTH, endpoint=False)

def jonswap_like_pulse(freq, direction, fp=0.1, dp=250.0, hs_scale=5.0):
    """Rough Gaussian-in-frequency, cos^4-in-direction bump -- not a real
    JONSWAP spectrum, just something with an obvious peak to look at."""
    freq_shape = np.exp(-((freq - fp) ** 2) / (2 * (0.01 ** 2)))
    dtheta = np.deg2rad(((direction - dp + 180) % 360) - 180)
    dir_shape = np.clip(np.cos(dtheta / 2), 0, None) ** 4
    return hs_scale * np.outer(freq_shape, dir_shape)  # (NK, NTH)


def calm_spectrum(freq, direction, floor=1e-4):
    """Small flat baseline -- not exactly 0 (physically implausible) or
    exactly the _FillValue, just a near-nothing sea state."""
    return floor * np.ones((len(freq), len(direction)))


# Time axis: hourly 00:00-12:00 Jan 1, then daily through Feb 1 (past the
# run end, so boundary forcing never runs out mid-run).
hourly = np.arange("2020-01-01T00", "2020-01-01T13", dtype="datetime64[h]")
daily = np.arange("2020-01-02", "2020-02-02", dtype="datetime64[D]")
time = np.concatenate([hourly.astype("datetime64[ns]"), daily.astype("datetime64[ns]")])
nt = len(time)

pulse = jonswap_like_pulse(freq, direction)
calm = calm_spectrum(freq, direction)

# West point: pulse held for the first 6 hourly records, calm after
# (records 6->7 give a 1 h linear ramp-down inside the model).
pulse_hours = 6
efth_west = np.stack([pulse if i < pulse_hours else calm for i in range(nt)])
# East point: calm the whole time.
efth_east = np.stack([calm for _ in range(nt)])

out_dir = "."
write_ww3_boundary_spectrum(
    f"{out_dir}/ww3.point1_spec.nc", lat=36.05, lon=328.05,
    freq=freq, direction=direction, efth=efth_west, time=time,
)
write_ww3_boundary_spectrum(
    f"{out_dir}/ww3.point2_spec.nc", lat=36.05, lon=333.95,
    freq=freq, direction=direction, efth=efth_east, time=time,
)

spectra_paths = [
    "ww3.point1_spec.nc",
    "ww3.point2_spec.nc",
]
write_spec_list(out_dir, spectra_paths)
write_ww3_bounc_nml(out_dir)
print(f"{nt} time records: {time[0]} .. {time[-1]}")